# Part 2 · A mini crystal structure prediction  (~9 min)

Generate trial packings, relax each one, rank by energy. We start from the paracetamol molecule only and look for
Form I and Form II. Packings come from [PyXtal](https://github.com/MaterSim/PyXtal), energies from MACE-OFF.

In [ ]:
# @title 0 · Setup  (run this cell first; ~1-2 min on Colab)
import os, sys
REPO = "https://github.com/fmggggg/agentic-ai-for-functional-materials-discovery"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/csp-tutorial"):
        !git clone -q {REPO} /content/csp-tutorial
    %cd /content/csp-tutorial
    !pip -q install pyxtal mace-torch py3Dmol pymatgen rdkit "requests==2.32.4"
import csp_utils as cu
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})
print("ready")

## 2.1 Take the molecule (keep its conformer rigid)

In [ ]:
form1 = cu.load_cif(cu.DATA / "paracetamol_form_I.cif")
mol = cu.molecule_from_crystal(form1)
print(mol.composition, len(mol), "atoms")

## 2.2 Generate random crystals

Most organic crystals are in one of five space groups: P2₁/c, P1̄, P2₁2₁2₁, C2/c, P2₁.

In [ ]:
trial = cu.random_crystal(mol, sg=14, z=4, seed=1)   # P2_1/c, 4 molecules
print("density of this random guess:", round(cu.density(trial), 3), "g/cm3  (experiment: 1.33)")
cu.show(trial, supercell=(2, 2, 2))

In [ ]:
import time
plan = [(14, 4), (2, 2), (19, 4), (4, 2), (61, 8)]
t0 = time.time()
raw = [cu.random_crystal(mol, sg, z, seed=i) for i, (sg, z) in enumerate(plan * 6)]
print(f"{len(raw)} structures in {time.time()-t0:.1f} s")
plt.hist([cu.density(a) for a in raw], bins=15, color="#C9A27E")
plt.axvline(cu.density(form1), color="#893F0D", lw=2, label="Form I (exp)")
plt.xlabel("density (g/cm$^3$)"); plt.ylabel("count"); plt.legend(); plt.title("Random guesses are loosely packed");

## 2.3 Relax two of them

Atoms and cell both move. We stop after 100 steps, so these are not fully converged.

In [ ]:
res = []
for k in range(2):
    t0 = time.time()
    at = cu.relax(raw[k], steps=100)
    res.append(dict(id=k, sg=plan[k % 5][0], density_before=round(cu.density(raw[k]), 3),
                    density_after=round(cu.density(at), 3),
                    E_per_mol=round(cu.energy_per_molecule(at), 1), seconds=round(time.time() - t0)))
    print(res[-1])
pd.DataFrame(res)

> **Question 3.** A real CSP study relaxes 10⁴–10⁵ structures per molecule. At your speed, how long would that take?

## 2.4 The full landscape (precomputed)

About 220 random structures, relaxed beforehand with the same settings. Stars: the two experimental forms,
relaxed with MACE-OFF. Energies are relative to Form I.

In [ ]:
from ase.io import read
L = cu.load_landscape("landscape_random")
e_gas = cu.gas_phase_energy()
exp = {}
for n in ("I", "II"):
    at = read(cu.DATA / f"paracetamol_form_{n}_relaxed.extxyz")
    exp[n] = (cu.density(at), cu.energy_per_molecule(at) - e_gas)
E0 = exp["I"][1]
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(L.density, L.lattice_energy_kjmol - E0, s=10, c="#BDBDBD", label=f"random search ({len(L)})")
for n, mk in [("I", "*"), ("II", "D")]:
    ax.scatter(exp[n][0], exp[n][1] - E0, s=160, marker=mk, edgecolor="k", zorder=5, label=f"Form {n} (experiment, relaxed)")
ax.set_xlabel("density (g/cm$^3$)"); ax.set_ylabel("energy relative to Form I (kJ/mol)"); ax.set_ylim(-2, 40)
ax.legend(); ax.set_title("Paracetamol: random search vs experiment");

In [ ]:
rel = (L.lattice_energy_kjmol - E0).sort_values()
print("random structures matching Form I or II:", int(L.rms_form_I.notna().sum() + L.rms_form_II.notna().sum()), "of", len(L))
print("lowest random structure:", round(rel.iloc[0], 1), "kJ/mol above Form I")
print("random structures within 5 kJ/mol of Form I:", (rel < 5).sum())

> **Question 4.** Did the random search find Form I or Form II? Its best structure is less than 1 kJ/mol above Form I
> but is a different packing. What does that tell you about ranking?
>
> **Question 5.** Roughly how many random trials would you need to find Form I? Keep the answer for Part 3.